<a href="https://colab.research.google.com/github/acamogh/RAG_QUIZ-APP/blob/main/RAG_Quiz_application.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pdfplumber
import pdfplumber
import re
import json
from collections import Counter
from pathlib import Path

def extract_questions(pdf_path):
    questions = []
    with pdfplumber.open(pdf_path) as pdf:
        full_text = "\n".join([page.extract_text() or "" for page in pdf.pages])
        pages_text = [page.extract_text() or "" for page in pdf.pages]


    # Split into questions
    blocks = re.split(r'(Question \d+[\s\.:])', full_text, flags=re.IGNORECASE)
    question_blocks = []
    i = 0
    while i < len(blocks)-1:
        if re.match(r'Question \d+', blocks[i], re.IGNORECASE):
            full_q = blocks[i] + (blocks[i+1] if i+1 < len(blocks) else "")
            question_blocks.append(full_q)
        i += 1

    for block in question_blocks:
        block = block.strip()
        if len(block) < 100:
            continue

        # Find page number
        page_number = 1
        for p_num, p_text in enumerate(pages_text, start=1):
            if block[:80] in p_text:
                page_number = p_num
                break

        # Category Detection
        block_lower = block.lower()
        category = "Uncategorized"
        if "google cloud" in block_lower or "gcp" in block_lower:
            category = "Google Cloud's gen AI offerings"
        elif "fundamental" in block_lower:
            category = "Fundamentals of gen AI"
        elif "business strateg" in block_lower:
            category = "Business strategies for a successful gen AI solution"
        elif "technique" in block_lower or "improve" in block_lower or "prompt" in block_lower:
            category = "Techniques to improve gen AI model output"

        questions.append({
            "question_id": len(questions) + 1,
            "question": block.split('\n')[0].strip(),
            "full_text": block,
            "category": category,
            "page_number": page_number,
            "source_pdf": pdf_path.name
        })

    return questions

# ====================== RUN ALL PDFs ======================
all_questions = []
pdf_files = list(Path(".").glob("*.pdf")) + list(Path(".").glob("*.PDF"))

for pdf_file in pdf_files:
    q_list = extract_questions(pdf_file)
    all_questions.extend(q_list)

print(f"Total Questions Extracted: {len(all_questions)}")

cat_count = Counter(q["category"] for q in all_questions)
print("\nCategory Distribution:")
for cat, count in cat_count.most_common():
    print(f"  • {cat}: {count}")

with open("processed_questions.json", "w", encoding="utf-8") as f:
    json.dump(all_questions, f, indent=2)

print("\n✅ Saved to processed_questions.json (without answer field)")

In [ ]:
# ========================= STEP 2: Upload to Pinecone =========================


!pip install pinecone sentence-transformers -q

from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
import json
from google.colab import userdata

# 3. Get Pinecone API Key from Colab Secrets
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')

# 4. Connect to Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "genai-exam-questions"   # Name of your vector database

# 5. Create index if it doesn't exist
if index_name not in [idx.name for idx in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=384,                    # Size of each vector
        metric="cosine",                  # How to measure similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print("New index created")
else:
    print("Connected to existing index")

# 6. Connect to the index
index = pc.Index(index_name)

# 7. Load embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# 8. Load questions from Step 1
with open("processed_questions.json", "r", encoding="utf-8") as f:
    all_questions = json.load(f)

print(f"Starting upload of {len(all_questions)} questions...")

# 9. Convert questions into vectors
vectors = []
for i, q in enumerate(all_questions):
    text = q.get("full_text", q.get("question", ""))   # Use full text for better search
    embedding = embedder.encode(text).tolist()         # Convert text → numbers
    vectors.append({
        "id": f"q_{i}",                                # Unique ID
        "values": embedding,                           # The vector (numbers)
        "metadata": {                                  # Extra information
            "question": text[:500],
            "full_text": text,
            "category": q.get("category", "Uncategorized"),
            "page_number": q.get("page_number", 1),
            "source_pdf": q.get("source_pdf", "")
        }
    })

# 10. Upload to Pinecone in batches
batch_size = 40
for i in range(0, len(vectors), batch_size):
    index.upsert(vectors=vectors[i:i+batch_size])   # Send batch to Pinecone

print("✅ Step 2 Completed - All questions are now in Pinecone!")

# 11. Show statistics
stats = index.describe_index_stats()
print(stats)

In [246]:
import gradio as gr
import json
import random
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
from groq import Groq
from google.colab import userdata
import os

# Setup
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("genai-exam-questions")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
groq_client = Groq(api_key=GROQ_API_KEY)

current_data = None


def log_failure_to_db(question, failure_type, cluster):
    db_file = "revision_db.json"

    # Load existing or create new
    if os.path.exists(db_file):
        with open(db_file, "r") as f:
            data = json.load(f)
    else:
        data = {}

    # Segregate based on cluster (category)
    if cluster not in data:
        data[cluster] = []

    # Prevent duplicates
    if not any(q['question'] == question for q in data[cluster]):
        data[cluster].append({
            "question": question,
            "failure_type": failure_type
        })

    with open(db_file, "w") as f:
        json.dump(data, f, indent=2)

def get_question_from_groq(category="All"):
    global current_data
    try:
        filter_dict = {"category": category} if category != "All" else None

        # Diversify queries to prevent getting stuck on one page
        queries = ["generative ai", "google cloud AI", "gen AI business strategy", "prompt engineering techniques"]
        query = random.choice(queries)

        result = index.query(
            vector=embedder.encode(query).tolist(),
            top_k=5, # Get more candidates to ensure variety
            filter=filter_dict,
            include_metadata=True
        )

        if not result['matches']:
            return "No content found.", gr.update(choices=[]), "**Error:** No content."

        # Pick a random match from the top 5
        match = random.choice(result['matches'])
        meta = match['metadata']
        context = meta.get('full_text', meta.get('question', ''))

        prompt = f"""Create one challenging MCQ based on this knowledge base:
{context}

Return **only** valid JSON in this exact format:
{{"question": "...", "options": {{"A": "...", "B": "...", "C": "...", "D": "..."}}, "correct_answer": "A", "explanation": "..."}}"""

        response = groq_client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7
        )

        content = response.choices[0].message.content.strip()
        data = json.loads(content.replace("```json", "").replace("```", ""))
        current_data = data

        options = [f"{k}. {v}" for k, v in data["options"].items()]
        source_info = f"📖 **Source:** {meta.get('source_pdf', 'N/A')} | **Page:** {meta.get('page_number', 'N/A')}"

        return data["question"], gr.update(choices=options), source_info
    except Exception as e:
        return f"Error: {str(e)}", gr.update(choices=[]), ""

def check_answer(selected_option):
    global current_data
    if not current_data: return "Load a question first.", ""

    correct = current_data.get("correct_answer", "A")
    is_correct = selected_option.strip().upper().startswith(correct)

    if is_correct:
        return "✅ **Correct!**", current_data.get("explanation", "")
    else:
        # Step 2 & 3: Log failure and provide feedback
        # For now, we use a placeholder cluster; next, we will link the LLM to provide the cluster name
        log_failure_to_db(
            current_data['question'],
            "Conceptual Gap", # Placeholder until LLM integration
            "Fundamentals of gen AI" # Placeholder for category
        )
        return "❌ **Wrong!**", f"Logged to Revision Tab. Explanation: {current_data.get('explanation')}"

# UI
with gr.Blocks(title="Gen AI Exam") as demo:
    gr.Markdown("# 🧠 Google Cloud Gen AI Exam Simulator")

    # Categories list matched to your extraction logic
    cat_list = [
        "All",
        "Google Cloud's gen AI offerings",
        "Fundamentals of gen AI",
        "Business strategies for a successful gen AI solution",
        "Techniques to improve gen AI model output"
    ]

    with gr.Tab("📋 Full Exam"):
        load_btn = gr.Button("Load New Question", variant="primary")
        source_display = gr.Markdown(label="Reference")
        question_box = gr.Textbox(label="Question", lines=6)
        options_radio = gr.Radio(choices=[], label="Select your answer")
        submit_btn = gr.Button("Submit")
        feedback_box = gr.Markdown()
        explanation_box = gr.Markdown()

        load_btn.click(get_question_from_groq, outputs=[question_box, options_radio, source_display])
        submit_btn.click(check_answer, inputs=options_radio, outputs=[feedback_box, explanation_box])

    with gr.Tab("🔍 Category Practice"):
        category_drop = gr.Dropdown(choices=cat_list, value="All", label="Select Category")
        load_btn_cat = gr.Button("Load Question")
        source_display_cat = gr.Markdown()
        question_box_cat = gr.Textbox(label="Question", lines=6)
        options_radio_cat = gr.Radio(choices=[], label="Select your answer")
        submit_btn_cat = gr.Button("Submit")
        feedback_box_cat = gr.Markdown()
        explanation_box_cat = gr.Markdown()

        load_btn_cat.click(lambda c: get_question_from_groq(c), inputs=category_drop, outputs=[question_box_cat, options_radio_cat, source_display_cat])
        submit_btn_cat.click(check_answer, inputs=options_radio_cat, outputs=[feedback_box_cat, explanation_box_cat])

    with gr.Tab("📚 Smart Revision"):
        revision_view = gr.Markdown("### Your Revision History\nSelect a category to see where you struggle.")
        refresh_btn = gr.Button("Refresh Revision Data")
        # This will display the content of your revision database
        revision_content = gr.JSON(label="Revision Database")

        def load_revision_data():
            if not os.path.exists("revision_db.json"):
                return {"Message": "No mistakes yet! Keep practicing."}
            with open("revision_db.json", "r") as f:
                return json.load(f)

        refresh_btn.click(load_revision_data, outputs=revision_content)

demo.launch(share=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ad65d94f2e36f7beee.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
